# Filters And Selection

This notebook is the third step in the operator sequence. The first notebook introduced unary decompositions. The second notebook introduced composition and additive unions. Here we show how to keep only the subgraphs that match structural or semantic constraints.


## What filtering does

A decomposition often generates more associated subgraphs than you actually want. Filter operators act on the current image graph and discard image nodes whose associated subgraphs do not satisfy some condition.

In this notebook we will cover:

1. size filters on nodes and edges
2. connectedness filters
3. node-label and edge-label filters
4. deterministic subsampling with `filter_by_sampling(...)`


In [ ]:
from pathlib import Path
import sys


def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
            return candidate
    return start


repo_root = _find_repo_root(Path.cwd())
candidate_src = repo_root / 'src'
if candidate_src.exists():
    candidate_src_str = str(candidate_src)
    if candidate_src_str not in sys.path:
        sys.path.insert(0, candidate_src_str)


In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

import networkx as nx
from IPython.core.display import HTML
from warnings import simplefilter

simplefilter(action='ignore', category=FutureWarning)
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')


In [ ]:
from abstractgraph.graphs import graph_to_abstract_graph
from abstractgraph.display import display, display_decomposition_graph, display_graph, display_mappings
from abstractgraph.operators import *


def draw(graph, decomposition_function, *, nbits=11, size=(12, 6), n_elements_per_row=8):
    display_decomposition_graph(decomposition_function)
    ag = graph_to_abstract_graph(graph, decomposition_function=decomposition_function, nbits=nbits)
    display(ag, size=size)
    display_mappings(ag, n_elements_per_row=n_elements_per_row)
    return ag


## A toy graph with node labels and edge labels

To make label-based filters concrete, this version of the toy graph attaches `label` attributes to both nodes and edges.


In [ ]:
graph = nx.Graph()
graph.add_nodes_from([
    (0, {'label': 'C'}),
    (1, {'label': 'C'}),
    (2, {'label': 'O'}),
    (3, {'label': 'N'}),
    (4, {'label': 'C'}),
    (5, {'label': 'S'}),
    (6, {'label': 'C'}),
    (7, {'label': 'O'}),
])
graph.add_edges_from([
    (0, 1, {'label': 'ring'}),
    (1, 2, {'label': 'ring'}),
    (2, 3, {'label': 'ring'}),
    (3, 0, {'label': 'ring'}),
    (2, 4, {'label': 'bridge'}),
    (4, 5, {'label': 'ring'}),
    (5, 2, {'label': 'ring'}),
    (3, 6, {'label': 'branch'}),
    (6, 7, {'label': 'branch'}),
])

print(graph)
display_graph(graph)


## Filter by number of nodes

A common pattern is to generate a rich local decomposition and then keep only subgraphs within a target size band.


In [ ]:
df = compose(filter_by_number_of_nodes(number_of_nodes=(4, 6)), neighborhood(radius=2))
node_count_filtered_ag = draw(graph, df)


## Filter by number of edges

This looks similar, but it is a different constraint. Two subgraphs can have the same number of nodes while differing in edge density.


In [ ]:
df = compose(filter_by_number_of_edges(number_of_edges=(4, 5)), neighborhood(radius=2))
edge_count_filtered_ag = draw(graph, df)


## Node count versus edge count on paths

Paths make the distinction especially clear: a 2-edge path has 3 nodes, while a 3-edge path has 4 nodes.


In [ ]:
df = compose(filter_by_number_of_nodes(number_of_nodes=(3, 4)), path(number_of_edges=(2, 3)))
path_node_filtered_ag = draw(graph, df)


In [ ]:
df = compose(filter_by_number_of_edges(number_of_edges=(2, 2)), path(number_of_edges=(2, 3)))
path_edge_filtered_ag = draw(graph, df)


## Filter by connected components

Combination-style operators can produce either connected or disconnected associated subgraphs. This filter lets you separate those cases.


In [ ]:
df = compose(
    filter_by_number_of_connected_components(number_of_components=(1, 1)),
    combination(number_of_elements=(2, 2), distance=(1, 2)),
    edge(),
)
connected_combinations_ag = draw(graph, df)


In [ ]:
df = compose(
    filter_by_number_of_connected_components(number_of_components=(2, 2)),
    combination(number_of_elements=(2, 2), distance=(1, 2)),
    edge(),
)
disconnected_combinations_ag = draw(graph, df)


## Filter by node label

The next example keeps only cycles that contain at least one `N` node.


In [ ]:
df = compose(filter_by_node_label(must_have_one_of=['N']), cycle())
node_label_filtered_ag = draw(graph, df)


You can also use label filters to exclude chemistry or domain-specific symbols. Here we keep only radius-1 neighborhoods that do not contain `S`.


In [ ]:
df = compose(filter_by_node_label(cannot_have_any_in=['S']), neighborhood(radius=1))
node_label_exclusion_ag = draw(graph, df)


## Filter by edge label

Because the toy graph includes `ring`, `bridge`, and `branch` edge labels, we can filter subgraphs based on relation type as well.


In [ ]:
df = compose(filter_by_edge_label(must_have_one_of=['branch']), neighborhood(radius=1))
edge_label_filtered_ag = draw(graph, df)


## Sampling as a selection operator

Sometimes you do not want a semantic filter at all. You just want a smaller, deterministic subset for debugging or visualization. `filter_by_sampling(...)` is useful for that.


In [ ]:
df = compose(
    intersection_edges(),
    filter_by_sampling(n_samples=5, seed=7),
    add(node(), edge()),
)
sampled_ag = draw(graph, df)


## Summary

Filtering operators turn broad decompositions into targeted vocabularies. They are usually most useful after unary decomposition and before more expensive downstream analysis.

A natural next notebook is `04_binary_and_combination_operators.ipynb`, where the focus shifts from selecting existing subgraphs to building new ones from relationships between subgraphs.
